In [ ]:
FIGURE = 'Fig.S7'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
def draw_frequency(pathway, threshold):
    version = '01_show_all' if threshold == 0 else '02_frequency_greater_than5'
    from pathlib import Path
    import numpy as np
    import matplotlib
    matplotlib.use('Agg')
    OUT = Path(asset_path('asset_418824be8e3a'))
    focus_window_records = []
    import numpy as np
    import xarray as xr
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    import os
    import rasterio
    from rasterio.warp import reproject, Resampling
    from scipy import ndimage
    import pandas as pd
    import warnings
    warnings.filterwarnings('ignore')
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['mathtext.fontset'] = 'stixsans'
    plt.rcParams['pdf.fonttype'] = 42
    plt.rcParams['ps.fonttype'] = 42
    base_output_dir = str(OUTPUT_DIR / pathway)
    base_data_path = str(OUT / 'data/adapted_inputs' / pathway)
    mask_path = asset_path('asset_e2603c619a67')
    time_scale_ranges = ['1', '2-3', '4-6', '7-12', '13-18', '19-24']
    seasons = ['Spring']
    start_years = range(2001, 2012)
    time_windows = [(year, year + 9) for year in start_years]
    FREQUENCY_THRESHOLD = threshold
    timescale_colors = {'1': {'light': '#E6F3FF', 'dark': '#0476E7'}, '2-3': {'light': '#E0F7FA', 'dark': '#00b2cb'}, '4-6': {'light': '#E8F5E9', 'dark': '#00AA09'}, '7-12': {'light': '#F8EEC2', 'dark': '#FFD000'}, '13-18': {'light': '#F7D2BA', 'dark': '#FF6600'}, '19-24': {'light': '#F5D0D5', 'dark': '#F10726'}}
    combined_output_dir = base_output_dir
    os.makedirs(combined_output_dir, exist_ok=True)
    print(f"{'=' * 100}")
    print(f'🌈 creatingNaturestyle_gradient_frequency_overlay(show_frequency_only > {FREQUENCY_THRESHOLD})')
    print(f"{'=' * 100}")
    print(f"\n{'=' * 60}")
    print('📂 reading_mask')
    print(f"{'=' * 60}")
    with rasterio.open(mask_path) as mask_src:
        mask_data = mask_src.read(1)
        mask_transform = mask_src.transform
        mask_crs = mask_src.crs
    print(f'✅ mask_shape: {mask_data.shape}')

    def hex_to_rgb(hex_color):
        hex_color = hex_color.lstrip('#')
        return tuple((int(hex_color[i:i + 2], 16) / 255.0 for i in (0, 2, 4)))

    def interpolate_color(light_hex, dark_hex, value):
        light_rgb = hex_to_rgb(light_hex)
        dark_rgb = hex_to_rgb(dark_hex)
        r = light_rgb[0] + (dark_rgb[0] - light_rgb[0]) * value
        g = light_rgb[1] + (dark_rgb[1] - light_rgb[1]) * value
        b = light_rgb[2] + (dark_rgb[2] - light_rgb[2]) * value
        return (r, g, b)
    for season_idx, season in enumerate(seasons, 1):
        saved = np.load(INPUT_DIR / f'{pathway}_frequency.npz')
        assert saved['timescales'].tolist() == time_scale_ranges
        reference_lat = saved['lat']
        reference_lon = saved['lon']
        timescale_frequencies = {}
        for i, t in enumerate(time_scale_ranges):
            frequency = saved['frequency'][i]
            if frequency.max() > 0:
                timescale_frequencies[t] = {'frequency': frequency, 'max_freq': int(frequency.max()), 'total_pixels': int((frequency > 0).sum()), 'processed_windows': 11}
        if len(timescale_frequencies) > 0 and reference_lat is not None:
            print(f'\n  🎨 creatingNaturestyle_visualization(show_frequency_only > {FREQUENCY_THRESHOLD})...')
            lat = reference_lat.copy()
            if lat[0] < lat[-1]:
                lat = lat[::-1]
                display_flipped = True
            else:
                display_flipped = False
            fig_height = 4.0
            aspect_ratio = 2.2
            fig_width = fig_height * aspect_ratio
            fig, ax = plt.subplots(1, 1, figsize=(fig_width, fig_height), subplot_kw={'projection': ccrs.PlateCarree()})
            ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color='black', zorder=10)
            ax.add_feature(cfeature.BORDERS, linewidth=0.2, color='gray', zorder=10)
            ax.add_feature(cfeature.OCEAN, color='lightblue', alpha=0.3, zorder=0)
            ax.add_feature(cfeature.LAND, color='lightgray', alpha=0.2, zorder=0)
            for time_scale_range in reversed(time_scale_ranges):
                if time_scale_range not in timescale_frequencies:
                    continue
                frequency_data = timescale_frequencies[time_scale_range]['frequency']
                max_freq = timescale_frequencies[time_scale_range]['max_freq']
                if display_flipped:
                    frequency_display = np.flipud(frequency_data)
                else:
                    frequency_display = frequency_data
                frequency_display_filtered = np.where(frequency_display > FREQUENCY_THRESHOLD, frequency_display, 0)
                valid_freq = frequency_display_filtered[frequency_display_filtered > 0]
                if len(valid_freq) == 0:
                    continue
                max_freq_filtered = np.max(frequency_display_filtered)
                frequency_normalized = frequency_display_filtered / max_freq_filtered
                light_color = timescale_colors[time_scale_range]['light']
                dark_color = timescale_colors[time_scale_range]['dark']
                height, width = frequency_display_filtered.shape
                rgba_array = np.zeros((height, width, 4))
                for i in range(height):
                    for j in range(width):
                        if frequency_display_filtered[i, j] > 0:
                            freq_ratio = frequency_normalized[i, j]
                            r, g, b = interpolate_color(light_color, dark_color, freq_ratio)
                            rgba_array[i, j, 0] = r
                            rgba_array[i, j, 1] = g
                            rgba_array[i, j, 2] = b
                            rgba_array[i, j, 3] = 0.8
                        else:
                            rgba_array[i, j, 3] = 0
                im = ax.imshow(rgba_array, extent=[reference_lon.min(), reference_lon.max(), lat.min(), lat.max()], transform=ccrs.PlateCarree(), origin='upper', zorder=5)
            season_short_labels = {'Spring': 'MAM', 'Summer': 'JJA', 'Autumn': 'SON', 'Winter': 'DJF'}
            ax.set_title(f'{season_short_labels[season]}', fontsize=20, pad=8, fontweight='normal')
            gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, linewidth=0.3, alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xlabel_style = {'size': 10, 'weight': 'normal'}
            gl.ylabel_style = {'size': 10, 'weight': 'normal'}
            from matplotlib.patches import Rectangle
            legend_x = 0.015
            legend_y = 0.015
            legend_width = 0.18
            legend_height = 0.423
            legend_bg = Rectangle((legend_x, legend_y), legend_width, legend_height, transform=ax.transAxes, facecolor='white', edgecolor='#2C2C2C', linewidth=0.8, alpha=0.98, zorder=20)
            ax.add_patch(legend_bg)
            ax.text(legend_x + legend_width / 2, legend_y + legend_height - 0.015, ' ', transform=ax.transAxes, fontsize=12, fontweight='bold', ha='center', va='top', color='#2C2C2C', zorder=21)
            bar_height = 0.045
            bar_spacing = 0.012
            start_y = legend_y + legend_height - 0.08
            bar_width = 0.04
            bar_x = legend_x + 0.012
            freq_column_width = 0.035
            freq_column_right = legend_x + legend_width - 0.012
            freq_column_center_x = freq_column_right - freq_column_width / 2
            legend_ranges = list(time_scale_ranges)
            for idx, time_scale_range in enumerate(legend_ranges):
                if time_scale_range not in timescale_frequencies:
                    continue
                bar_y = start_y - idx * (bar_height + bar_spacing)
                light_color = timescale_colors[time_scale_range]['light']
                dark_color = timescale_colors[time_scale_range]['dark']
                n_segments = 20
                segment_width = bar_width / n_segments
                for seg_idx in range(n_segments):
                    seg_x = bar_x + seg_idx * segment_width
                    ratio = seg_idx / (n_segments - 1)
                    r, g, b = interpolate_color(light_color, dark_color, ratio)
                    seg_rect = Rectangle((seg_x, bar_y), segment_width, bar_height, transform=ax.transAxes, facecolor=(r, g, b), edgecolor='none', zorder=21)
                    ax.add_patch(seg_rect)
                border_rect = Rectangle((bar_x, bar_y), bar_width, bar_height, transform=ax.transAxes, facecolor='none', edgecolor='#2C2C2C', linewidth=0.5, zorder=22)
                ax.add_patch(border_rect)
                label_text = f'{time_scale_range}'
                text_x = bar_x + bar_width + 0.008
                text_y = bar_y + bar_height / 2
                ax.text(text_x, text_y, label_text, transform=ax.transAxes, fontsize=12, fontweight='normal', va='center', ha='left', color='#2C2C2C', zorder=23)
                max_freq = timescale_frequencies[time_scale_range]['max_freq']
                freq_text = f'{max_freq}'
                ax.text(freq_column_center_x, text_y, freq_text, transform=ax.transAxes, fontsize=12, fontweight='normal', va='center', ha='center', color='#555555', family='monospace', zorder=23)
            ax.text(legend_x + legend_width / 2, legend_y + 0.008, 'Max frequency', transform=ax.transAxes, fontsize=9, style='italic', ha='center', va='bottom', color='#666666', zorder=21)
            plt.tight_layout()
            output_filename = f'nature_style_gradient_{season}_all_frequencies.png'
            output_path = os.path.join(combined_output_dir, output_filename)
            plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
            print(f'  ✅ saved: {output_filename}')
            plt.close(fig)
            stats_data = []
            for time_scale_range in time_scale_ranges:
                if time_scale_range in timescale_frequencies:
                    info = timescale_frequencies[time_scale_range]
                    freq_data = info['frequency']
                    filtered_pixels = np.sum(freq_data > FREQUENCY_THRESHOLD)
                    filtered_freq = freq_data[freq_data > FREQUENCY_THRESHOLD]
                    stats_data.append({'timescale': time_scale_range, 'light_color': timescale_colors[time_scale_range]['light'], 'dark_color': timescale_colors[time_scale_range]['dark'], 'total_pixels': info['total_pixels'], 'filtered_pixels': filtered_pixels, 'filter_ratio': f"{filtered_pixels / info['total_pixels'] * 100:.1f}%", 'max_frequency': info['max_freq'], 'max_frequency_filtered': np.max(filtered_freq) if len(filtered_freq) > 0 else 0, 'processed_windows': info['processed_windows'], 'avg_frequency': np.mean(info['frequency'][info['frequency'] > 0]), 'avg_frequency_filtered': np.mean(filtered_freq) if len(filtered_freq) > 0 else 0})
            if stats_data:
                df_stats = pd.DataFrame(stats_data)
                csv_filename = f'nature_style_statistics_{season}_all_frequencies.csv'
                csv_path = os.path.join(combined_output_dir, csv_filename)
                df_stats.to_csv(csv_path, index=False, encoding='utf-8-sig')
                print(f'  ✅ statistics_table_saved: {csv_filename}')
        else:
            print(f'  ⚠️ season {season} no_valid_data')
    print(f"\n{'=' * 100}")
    print(f'✅ completed!show_frequency_only > {FREQUENCY_THRESHOLD} regions')
    print(f"{'=' * 100}")
for pathway in ['SPEI', 'SSMIrz']:
    draw_frequency(pathway, 0)
from PIL import Image
images = [Image.open(path).convert('RGB') for path in sorted(OUTPUT_DIR.rglob('nature_style_gradient_*.png'))]
if images:
    canvas = Image.new('RGB', (sum((im.width for im in images)), max((im.height for im in images))), 'white')
    x = 0
    for im in images:
        canvas.paste(im, (x, 0))
        x += im.width
    canvas.save(OUTPUT_DIR / 'Fig.frequency_600dpi.png', dpi=(600, 600))
